# Notebook 2 - Bronze to Silver

# **Notebook 2 - Bronze -> Silver (limpeza e padronizacao)**
**Projeto:** CineData Analytics | Rocket Lab 2026 | **PySpark + Delta**
**Camada de transformacao:** Silver

---
## Objetivo
Construir a camada **Silver**: dados **limpos, tipados e padronizados**, prontos para a modelagem
dimensional (Gold). Mantemos os nomes de colunas em **portugues** e garantimos **tipagem correta**.

**Meta:** nao duplicar o grao, preservar a cotacao do dolar (BRL) como serie continua e
aplicar as regras de negocio de **escala** (notas 0-10, contagens nao negativas).


---
| Tabela Silver | Origem (Bronze) |
|---|---|
| `silver.tb_info_filmes` | `bronze.tb_movies_info` |
| `silver.tb_financeiro_filmes` | `bronze.tb_movies_financials` |
| `silver.tb_metricas_engajamento` | `bronze.tb_movies_metrics` |
| `silver.tb_avaliacoes_usuarios` | `bronze.tb_movies_reviews` |
| `silver.tb_generos` | `bronze.tb_credits_and_tags` (coluna `genres`) |
| `silver.tb_pessoas_empresas` | `bronze.tb_credits_and_tags` (cast/directors/writers/companies) |
| `silver.tb_cotacao_dolar` | `bronze.tb_cotacao_dolar` (forward fill) |


In [ ]:
# ============================================================
# 1) Ambiente Silver
# ============================================================
silver_db = "silver"
bronze_db = "bronze"
delta_format = "delta"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {silver_db}")
print(f"Database `{silver_db}` pronto.")


In [ ]:
# ============================================================
# 1.1) UDF segura de conversao numerica (higieniza moeda/contagem)
# ============================================================
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

def limpa_numero(texto, tipo=DoubleType()):
    return (F.udf(lambda v:
        None if v is None else (
            _limpa_interno(str(v)) if (_limpa_interno := lambda s: None) else None
        ), tipo))

# Implementacao direta, sem lambda recursivo (mais legivel e testavel)
import re
def _parse_numero(v):
    if v is None: return None
    s = str(v).strip()
    if s.lower() in ("unknown","n/a","na","nan","none","null","-",""):
        return None
    # remove simbolos de moeda e separadores de milhar
    s = re.sub(r"[R$r$US\s]", "", s)          # tira R$, US, espacos
    s = s.replace(".", "").replace(",", ".")  # milhar -> decimal
    try:
        return float(s)
    except Exception:
        return None

udf_num = F.udf(_parse_numero, DoubleType())
print("UDF de conversao numerica pronta.")


In [ ]:
# ============================================================
# 2) tb_info_filmes: dedup por id (mais recente) + tipos
# ============================================================
from pyspark.sql import Window, functions as F

w = Window.partitionBy("id").orderBy(F.desc("ingestion_datetime"))
df_info = (spark.table(f"{bronze_db}.tb_movies_info")
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .drop("rn")
    .withColumn("id_filme", F.col("id").cast("bigint"))
    .withColumnRenamed("title","titulo")
    .withColumnRenamed("original_title","titulo_original")
    .withColumnRenamed("release_date","data_lancamento")
    .withColumnRenamed("runtime","duracao_minutos")
    .withColumnRenamed("original_language","idioma_original")
    .withColumnRenamed("status","status_filme")
    .withColumnRenamed("overview","sinopse")
    .withColumnRenamed("tagline","frase_divulgacao"))

df_info.write.mode("overwrite").format(delta_format) \
      .saveAsTable(f"{silver_db}.tb_info_filmes")
print(f"[SILVER] tb_info_filmes: {df_info.count():,}")


In [ ]:
# ============================================================
# 3) Data multi-formato -> ISO (YYYY-MM-DD) + ano_lancamento
# ============================================================
formatos = ["yyyy-MM-dd","MM-dd-yyyy","dd/MM/yyyy","yyyy/MM/dd",
            "MM/dd/yyyy","dd-MM-yyyy","yyyyMMdd"]

def converte_data(col_in="release_date"):
    exprs = [F.to_date(F.col(col_in), f) for f in formatos]
    return F.coalesce(*exprs)

df_info = (df_info
  .withColumn("data_lancamento", converte_data("data_lancamento"))
  .withColumn("ano_lancamento", F.year("data_lancamento"))
  .withColumn("duracao_minutos",
      F.col("duracao_minutos").cast("int")))
print("[SILVER] data padronizada OK")


In [ ]:
# ============================================================
# 4) tb_financeiro_filmes (origem tb_movies_financials)
# ============================================================
# "Unknown" -> NULL antes da conversao; higieniza $ e milhar
df_fin = (spark.table(f"{bronze_db}.tb_movies_financials")
    .withColumn("orcamento_usd",  udf_num("budget"))
    .withColumn("receita_usd",    udf_num("revenue"))
    .drop("budget","revenue"))

df_fin.write.mode("overwrite").format(delta_format) \
      .saveAsTable(f"{silver_db}.tb_financeiro_filmes")
print(f"[SILVER] tb_financeiro_filmes: {df_fin.count():,}")


In [ ]:
# ============================================================
# 5) tb_metricas_engajamento (origem tb_movies_metrics)
# ============================================================
# TIPAGEM SEGURA: textos/coluna-shift -> NULL (nunca linha corrompida)
df_met = (spark.table(f"{bronze_db}.tb_movies_metrics")
    .withColumn("popularidade", udf_num("popularity"))
    .withColumn("nota_media_tmdb",
        F.when((F.col("vote_average")>=0) & (F.col("vote_average")<=10),
               F.col("vote_average")).otherwise(None))
    .withColumn("qtd_votos_tmdb",
        F.when(F.col("vote_count")>=0, F.col("vote_count")).otherwise(None))
    .withColumn("nota_media_imdb",
        F.when((F.col("averageRating")>=0) & (F.col("averageRating")<=10),
               F.col("averageRating")).otherwise(None))
    .withColumn("qtd_votos_imdb",
        F.when(F.col("numVotes")>=0, F.col("numVotes")).otherwise(None)))

df_met.write.mode("overwrite").format(delta_format) \
      .saveAsTable(f"{silver_db}.tb_metricas_engajamento")
print(f"[SILVER] tb_metricas_engajamento: {df_met.count():,}")


In [ ]:
# ============================================================
# 6) tb_avaliacoes_usuarios (origem tb_movies_reviews)
# ============================================================
df_rev = (spark.table(f"{bronze_db}.tb_movies_reviews")
    .select("id","nome","nota" ,"comentario")
    .withColumnRenamed("id","id_filme")
    .withColumnRenamed("nome","nome_usuario")
    .withColumnRenamed("nota","nota_usuario")
    .withColumnRenamed("comentario","comentario_usuario")
    # dedup integral
    .distinct()
    # escala 0-10
    .withColumn("nota_usuario",
        F.when((F.col("nota_usuario")>=0)&(F.col("nota_usuario")<=10),
               F.col("nota_usuario")).otherwise(None))
    # comentario em branco -> padronizado
    .withColumn("comentario_usuario",
        F.when((F.trim(F.col("comentario_usuario"))=="") |
               F.col("comentario_usuario").isNull(),
               F.lit("Sem comentario"))
         .otherwise(F.col("comentario_usuario"))))

df_rev.write.mode("overwrite").format(delta_format) \
      .saveAsTable(f"{silver_db}.tb_avaliacoes_usuarios")
print(f"[SILVER] tb_avaliacoes_usuarios: {df_rev.count():,}")


In [ ]:
# ============================================================
# 7) tb_generos (origem credits_and_tags.genres) - explode
# ============================================================
# Trata separador: ';' e ',' -> normaliza p/ ',' antes do split
df_gen = (spark.table(f"{bronze_db}.tb_credits_and_tags")
    .select("id",
        F.regexp_replace("genres","[;|]"," ").alias("genres_ok"))
    .withColumn("genero",
        F.explode(F.split("genres_ok","\\s+,\\s*|,\\s*")))
    .filter(F.trim("genero") != "")
    .withColumn("nome_genero", F.initcap(F.trim("genero")))
    .select("id","nome_genero")
    .distinct())

df_gen.write.mode("overwrite").format(delta_format) \
      .saveAsTable(f"{silver_db}.tb_generos")
print(f"[SILVER] tb_generos: {df_gen.count():,}")


In [ ]:
# ============================================================
# 8) tb_pessoas_empresas (origem credits_and_tags) - entidades
# ============================================================
from pyspark.sql.types import StringType

def explode_entidade(col_origem, tipo):
    return (spark.table(f"{bronze_db}.tb_credits_and_tags")
        .select("id",
            F.regexp_replace(F.coalesce(F.col(col_origem), F.lit("")),
                             "[\[\]'\"]","").alias("ent_")))
        .select("id", F.explode(F.split("ent_",","|;")).alias("ent_raw"))
        .filter(F.trim("ent_raw") != "")
        .withColumn("nome_entidade", F.initcap(F.trim("ent_raw")))
        .withColumn("tipo_entidade", F.lit(tipo))
        .select("id","nome_entidade","tipo_entidade"))

df_ent = (explode_entidade("cast","Ator")
    .union(explode_entidade("directors","Diretor"))
    .union(explode_entidade("writers","Roteirista"))
    .union(explode_entidade("production_companies","Produtora"))
    .distinct())

df_ent.write.mode("overwrite").format(delta_format) \
      .saveAsTable(f"{silver_db}.tb_pessoas_empresas")
print(f"[SILVER] tb_pessoas_empresas: {df_ent.count():,}")


In [ ]:
# ============================================================
# 9) tb_cotacao_dolar - Forward Fill (serie continua)
# ============================================================
# O PTAX nao produz fim de semana/feriado -> preenchemos com o
# ultimo dia util via forward fill dentro de uma serie temporal.
w_fill = (F.col("data_cotacao"))
df_cot = (spark.table(f"{bronze_db}.tb_cotacao_dolar")
    .select("data_cotacao","cotacao_compra_brl")
    .orderBy("data_cotacao"))

# Gera sequencia diaria (calendar) + forward fill do valor
from pyspark.sql.types import DateType
cal = spark.range(0, 730).select(
    F.date_add(F.to_date(F.lit("2024-01-01")), F.col("id")).alias("dia"))
fill = (cal.join(df_cot,
          F.col("dia") >= F.col("data_cotacao"), "left")
  .groupBy("dia")
  .agg(F.max("data_cotacao").alias("ult_util"))
  .join(df_cot, "ult_util" == "data_cotacao", "left"))

df_saida = (fill.select(
    "dia","cotacao_compra_brl")
  .withColumnRenamed("dia","data_cotacao"))

df_saida.write.mode("overwrite").format(delta_format) \
      .saveAsTable(f"{silver_db}.tb_cotacao_dolar")
print(f"[SILVER] tb_cotacao_dolar (forward fill): {df_saida.count():,}")


---
## Validacao da camada Silver


In [ ]:
# ============================================================
# 10) Conferencia
# ============================================================
for t in ["tb_info_filmes","tb_financeiro_filmes","tb_metricas_engajamento",
          "tb_avaliacoes_usuarios","tb_generos","tb_pessoas_empresas",
          "tb_cotacao_dolar"]:
    try:
        print(f"  {silver_db}.{t}: {spark.table(f'{silver_db}.{t}').count():,}")
    except Exception as e:
        print(f"  {silver_db}.{t}: erro -> {e}")
